# film

> A demo as a narrated explainer video: rupaka plans it, the voice of this app speaks it, and manimgl draws it.

A film is a short video of one demo. It asks the gate question, counts down so that you can predict, moves the input, and shows the answer.
You can watch it on the concept page, or give it to a learner who is not at the app.

- rupaka makes the film data (`rupaka.film.plan`) and draws it with manimgl (`rupaka.film.render`). The scene only draws data, so a demo from a model cannot run code.
- abhyasa speaks each line with the voice of the settings (`abhyasa.voice.speak`), and times each beat to its line.
- The video goes in the assets folder of the vault. A `film` document on the learning shelf keeps its path, its length and its lines, so search finds it.

A film needs the extra `rupaka[film]` and ffmpeg. With no voice, the film has subtitles and no sound.

In [ ]:
#| default_exp film

In [ ]:
#| hide
import os, tempfile
os.environ.setdefault('LITESEARCH_CACHE', tempfile.mkdtemp(prefix='ls-'))   # and litesearch does not wait on the shared lock of another run
os.environ['ABHYASA_CONFIG'] = tempfile.mkdtemp()   # the settings go to a temporary folder, not to your real settings

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re, json, time
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor
from fastcore.utils import *
import rupaka
from rupaka.film import plan, narration, timed, track, render, can_film
from abhyasa.store import Learning, _src
from abhyasa.voice import speak, wav_seconds
from abhyasa.jobs import track as track_job
import abhyasa.demo   # Learning.demo

## Where films go

The videos go in the folder `films` in the assets folder of the vault. A test vault has its own folder.

In [ ]:
#| export
def films_dir(lr:Learning) -> Path:
    "The folder of the videos, beside the vault."
    p = Path(lr.vault.assets())/'films'
    p.mkdir(parents=True, exist_ok=True)
    return p

A new test vault has an empty films folder.

In [ ]:
from abhyasa.store import temp_learning
lr = temp_learning()
test_eq(list(films_dir(lr).iterdir()), [])

## The voice

`voice` speaks each line and gives the WAV and the seconds of each one. When no voice speaks here (the browser voice, or no model), it gives `(None, None)`, and the film has no sound.

In [ ]:
#| export
def _say(text): return speak(text)[0]

def voice(lines, say=None) -> tuple:
    "The WAV and the seconds of each line, from `say` or the voice of the settings; `(None, None)` when no voice speaks here."
    say = say or _say
    try: wavs = [say(l) if l else None for l in lines]
    except Exception: return None, None
    return wavs, [wav_seconds(w) if w else 0. for w in wavs]

A test voice gives one second of sound for each line. A voice that fails gives no sound.

In [ ]:
from rupaka.film import _wav
def one_sec(t): return _wav(b'\0\0' * 24000, 24000)
wavs, secs = voice(['A line.', 'Another line.'], one_sec)
test_eq(secs, [1., 1.])
def no_voice(t): raise ValueError('The browser speaks with its own voice.')
test_eq(voice(['A line.'], no_voice), (None, None))

## The film shelf

`put_film` keeps one `film` document for each demo. Its text is the voice lines, so search finds the film. A new film of the same demo takes the place of the old one.

In [ ]:
#| export
def _meta(r): return AttrDict({**json.loads(r['meta'] or '{}'), 'doc_id': r['id']})

@patch
def put_film(self:Learning, d, path, p:dict, voiced:bool) -> AttrDict:
    "Keep the film of demo `d` at `path`, made from plan `p`."
    m = dict(slug=d.slug, concept=d.concept, path=str(path), title=p['title'], secs=round(sum(b['secs'] for b in p['beats']), 1),
             beats=len(p['beats']), voiced=voiced, at=time.time(), rupaka=rupaka.__version__)
    r = self.v.add('\n\n'.join([f'Video of the demo {d.slug} for {d.concept}', *narration(p)]), title=f"Video: {p['title']}",
                   source=_src('film', d.slug), kind='film', meta=m)
    if not r.get('changed'): self.v.set_meta(r['doc_id'], **m)
    return self.film(d.slug)

`film` gives the film of a demo when its video is on the disk. `films` gives the films of a concept, newest first.

In [ ]:
#| export
@patch
def film(self:Learning, slug:str) -> AttrDict|None:
    "The film of demo `slug`; None when there is none, or when its video is not on the disk."
    r = first(self.v.t.docs(where='source=?', where_args=[_src('film', slug)]))
    return f if r and Path((f := _meta(r)).path).exists() else None

@patch
def films(self:Learning, concept:str=None) -> L:
    "The films of `concept` (or all films) whose video is on the disk, newest first."
    rs = self._docs('film', *(("json_extract(meta, '$.concept')=?", [concept]) if concept else ()))
    return L(sorted((f for f in map(_meta, rs) if Path(f.path).exists()), key=lambda f: -f.get('at', 0)))

## Make a film

`make_film` plans the film of a demo, speaks it, times it, and draws it. `draw` is `rupaka.film.render`; a test gives its own.

In [ ]:
#| export
def make_film(lr:Learning, slug:str, voiced:bool=True, say=None, res:str='1920x1080', palette:str='board', draw=None) -> AttrDict:
    "Make the film of demo `slug` and keep it. With `voiced`, the voice of the settings (or `say`) speaks each line."
    if (d := lr.demo(slug)) is None: raise KeyError(f'No demo {slug!r}.')
    p = plan(d.spec, palette=palette)
    wavs, secs = voice(narration(p), say) if voiced else (None, None)
    if secs: p = timed(p, secs)
    out = films_dir(lr)/f'{d.slug}.mp4'
    (draw or render)(p, out, track(wavs, [b['secs'] for b in p['beats']]) if wavs else None, res=res)
    return lr.put_film(d, out, p, bool(wavs))

A test draw writes a small file. The film keeps the seconds of the voice, and search finds its lines.

In [ ]:
from rupaka.spec import gallery
lr.put('Softmax temperature', 'Softmax divides each score by a temperature first.', 'ml')
d = lr.put_demo(gallery()['softmax-temperature'], 'softmax-temperature')
def fake_draw(p, out, wav=None, res=''): Path(out).write_bytes(b'mp4' + (wav or b'')); return out
f = make_film(lr, d.slug, say=one_sec, draw=fake_draw)
test_eq((f.voiced, f.concept, f.beats), (True, 'softmax-temperature', 4))
assert f.secs > 4 and Path(f.path).exists()
test_eq(lr.films('softmax-temperature').attrgot('slug'), [d.slug])

With no voice, the film is kept with no sound. The same demo keeps one film.

In [ ]:
f2 = make_film(lr, d.slug, voiced=False, draw=fake_draw)
test_eq(f2.voiced, False)
test_eq(len(lr.films()), 1)
test_fail(lambda: make_film(lr, 'nope', draw=fake_draw), contains='No demo')

## A film job

A film takes a minute or more, so the app makes it on a job. The job has a row, and the activity chip shows it.
One film draws at a time: manimgl uses much of the CPU.

In [ ]:
#| export
_POOL = ThreadPoolExecutor(1)

def film_key(slug:str) -> str: return f'film:{slug}'

def _why(e) -> str: return f'The video did not draw: {str(e).splitlines()[0][:160]}' if str(e) else 'The video did not draw.'

def queue_film(tutor, slug:str, **kw):
    "Start a job that makes the film of demo `slug`, or give the job that runs now."
    k, lr = film_key(slug), tutor.learning
    if (j := tutor.jobs.get(k)) is not None and not j.done(): return j
    d = lr.demo(slug)
    j = tutor.jobs[k] = track_job(lr, k, lambda: _POOL.submit(make_film, lr, slug, **kw), 'film', ok=lambda f: dict(said='The video is ready.'), why=_why,
                                  title=f"Video: {d.spec.get('title') if d else slug}", said='Drawing the video.', url=f'/c/{d.concept}' if d else '', obj=k)
    return j

def film_state(tutor, slug:str) -> str:
    "'running', 'failed', 'done' or 'none' for the film job of `slug`."
    j = tutor.jobs.get(film_key(slug))
    return 'none' if j is None else 'running' if not j.done() else 'failed' if j.exception() else 'done'

A job with the test draw ends with the film on the shelf, and its row says done.

In [ ]:
import time
class T: pass
t = T(); t.learning, t.jobs = lr, {}
j = queue_film(t, d.slug, voiced=False, draw=fake_draw)
j.result(timeout=30)
test_eq(film_state(t, d.slug), 'done')
for _ in range(50):   # the row ends in a callback, just after the result
    if lr.job(film_key(d.slug)).state != 'running': break
    time.sleep(.1)
test_eq(lr.job(film_key(d.slug)).state, 'done')

## Subtitles for the page

A browser reads WebVTT, not SubRip. `vtt` changes the subtitles that rupaka writes beside the video.

In [ ]:
#| export
def vtt(srt:str) -> str:
    "SubRip subtitles as WebVTT."
    return 'WEBVTT\n\n' + re.sub(r'(\d\d:\d\d:\d\d),(\d\d\d)', r'\1.\2', srt)

In [ ]:
test_eq(vtt('1\n00:00:00,000 --> 00:00:02,500\nA line.\n'), 'WEBVTT\n\n1\n00:00:00.000 --> 00:00:02.500\nA line.\n')

This slow test draws a real film when manimgl is here. Run it with `nbdev-test --flags slow`.

In [ ]:
#| slow
if can_film():
    f = make_film(lr, d.slug, voiced=False, res='640x360')
    assert Path(f.path).stat().st_size > 10_000